In [ ]:
!pip install flash-attn sympy math_verify pylatexenc


In [ ]:
from vllm import LLM, SamplingParams

# Create an LLM.
llm = LLM(model='Qwen/Qwen2.5-Math-1.5B')

# Read fine tuned model from local
# llm = LLM(model='sft_model/')

In [ ]:
question = "Simplify $(3-i)(6+2i)$." 

# Sample prompts.
prompts = [
    f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>""",
]

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["\n"]
)

sampling_params.stop = ["</answer>"]
sampling_params.include_stop_str_in_output = True

# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.generate(prompts, sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")

In [ ]:
from drgrpo_grader import r1_zero_reward_fn


In [ ]:
r1_zero_reward_fn(generated_text, '20')


In [ ]:
from typing import Callable

def evaluate_vllm(
  vllm_model: LLM,
  reward_fn: Callable[[str, str], dict[str, float]],
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) :
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)
  # format_reward = 0.
  # answer_reward = 0.
  # reward = 0.
  output_rewards = []
      
  for index, output in enumerate(outputs):
    prompt = output.prompt
    generated_text = output.outputs[0].text
    rewards = reward_fn(generated_text, answers[index])
    output_rewards.append(rewards)
    # format_reward += rewards['format_reward']
    # answer_reward += rewards['answer_reward']
    # reward += rewards['reward']
    # if rewards['reward'] == 1.0:
    #     print(f"Prompt: {prompt!r}")
    #     print(f"Generated text: {generated_text!r}")
    #     print(f"Answer: {answers[index]}")
    #     print(f"Reward: {rewards}")
    #     print('*****************')

  return output_rewards

In [ ]:
import pandas as pd
from tqdm import tqdm
import random 

df = pd.read_parquet("math_12k_test.parquet")

total_format_reward = 0.
total_answer_reward = 0.
total_reward = 0.
prompts = []
answers = []

for index, row in tqdm(df.iterrows()):
    question = row['problem']
    answer = row['solution']
    prompt = f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>"""
    prompts.append(prompt)
    answers.append(answer)

    if (index + 1) % 240 == 0:
        # Initialize batch counters
        batch_format_reward = 0
        batch_answer_reward = 0
        batch_total_reward = 0
        
        shots = [evaluate_vllm(llm, r1_zero_reward_fn, prompts, answers, sampling_params)]
        # for i in range(4):
        #     shots.append(evaluate_vllm(llm, r1_zero_reward_fn, prompts, answers, sampling_params))

        # for r0, r1, r2, r3 in zip(shots[0], shots[1], shots[2], shots[3]):
            # if random.randint(1,20) == 20:
            #     print(f"Rewards: r0={r0}, r1={r1}, r2={r2}, r3={r3}")
            
            # if (r0['format_reward'] == 1.0 or r1['format_reward'] == 1.0 or r2['format_reward'] == 1.0 or r3['format_reward'] == 1.0):
            #     batch_format_reward += 1
            # if (r0['answer_reward'] == 1.0 or r1['answer_reward'] == 1.0 or r2['answer_reward'] == 1.0 or r3['answer_reward'] == 1.0):
            #     batch_answer_reward += 1
            # if (r0['reward'] == 1.0 or r1['reward'] == 1.0 or r2['reward'] == 1.0 or r3['reward'] == 1.0):
            #     batch_total_reward += 1

        for r0 in shots[0]:
            if (r0['format_reward'] == 1.0):
                batch_format_reward += 1
            if (r0['answer_reward'] == 1.0):
                batch_answer_reward += 1
            if (r0['reward'] == 1.0):
                batch_total_reward += 1
        
        # Add batch results to totals
        total_format_reward += batch_format_reward
        total_answer_reward += batch_answer_reward
        total_reward += batch_total_reward
        
        print(f"Batch reward: {batch_total_reward}")
        print(f"Total reward so far: {total_reward}")
        prompts = []
        answers = []

print(f"Total format reward: {total_format_reward}")
print(f"Total answer reward: {total_answer_reward}")
print(f"Total reward: {total_reward}")